In [1]:
from typing import List,Sequence,TypedDict,Annotated
from dotenv import load_dotenv,find_dotenv
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import END,StateGraph
from langgraph.graph.message import add_messages
from chains import reflection_chain,generation_chain

load_dotenv(find_dotenv())


class AgentState(TypedDict):
    messages:Annotated[Sequence[BaseMessage],add_messages]

graph = StateGraph(AgentState)

REFLECT = "reflect"
GENERATE = "generate"

def generate_node(state:AgentState):
    response =  generation_chain.invoke({
        "messages":state["messages"]
    })
    print(response.content)
    return response

def reflect_node(state:AgentState):
    response =  reflection_chain.invoke(
        {
            "messages":state["messages"]
        }
    )
    print(response.content)
    return {"messages":[HumanMessage(content=response.content)]}

def should_continue(state):
    if len(state["messages"]) > 4:
        return "end"
   
    return "continue"

graph.add_node(GENERATE,generate_node)
graph.add_node(REFLECT,reflect_node)

graph.set_entry_point(GENERATE)

graph.add_conditional_edges(GENERATE,should_continue,{
    "end":END,
    "continue":REFLECT
})
graph.add_edge(REFLECT,GENERATE)








c:\Users\Josh\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
tweet = graph.compile()

tweet.invoke([HumanMessage(cotent = "AI agent taking over human kind")])

ValidationError: 2 validation errors for HumanMessage
content.str
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type
content.list[union[str,dict[any,any]]]
  Input should be a valid list [type=list_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.11/v/list_type